In [1]:
from unet.unet_model import *
import pandas as pd
from pathlib import Path

In [2]:
data_base = Path("data")

In [3]:
df_data = pd.read_csv(data_base/'attack.csv')
df_label = pd.read_csv(data_base/'label.csv')
df_data.shape, df_label.shape

((4985, 304), (4985, 304))

In [4]:
data_tensor = torch.tensor(df_data.values).unsqueeze(1).float()
label_tensor = torch.tensor(df_label.values)

torch.isnan(data_tensor).any(), data_tensor.shape, torch.isnan(label_tensor).any(), label_tensor.shape

(tensor(False),
 torch.Size([4985, 1, 304]),
 tensor(False),
 torch.Size([4985, 304]))

In [15]:
data_tensor[0].unsqueeze(0).size(-1)

304

In [5]:
model = UNet_1D(n_channels=1, n_classes=2, bilinear=False)
criterion = nn.CrossEntropyLoss() if model.n_classes > 1 else nn.BCEWithLogitsLoss()

In [6]:
masks_pred = model(data_tensor)
torch.isnan(masks_pred).any(), masks_pred.shape

(tensor(False), torch.Size([4985, 2, 304]))

In [7]:
from utils.dice_score import dice_loss

In [8]:
masks_pred.dtype, label_tensor.dtype

(torch.float32, torch.int64)

In [9]:
F.softmax(masks_pred, dim=1).float().shape

torch.Size([4985, 2, 304])

In [10]:
if model.n_classes == 1:
    loss = criterion(masks_pred.squeeze(1), label_tensor)
    loss += dice_loss(F.sigmoid(masks_pred.squeeze(1)), label_tensor, multiclass=False)
else:
    loss = criterion(masks_pred, label_tensor)
    loss += dice_loss(
        F.softmax(masks_pred, dim=1).float(),
        F.one_hot(label_tensor, model.n_classes).permute(0, 2, 1).float(),
        multiclass=True
    )

In [11]:
loss

tensor(1.1474, grad_fn=<AddBackward0>)